In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# old

In [ ]:
# =============================================================================
# 1. SETUP & IMPORTS
# =============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
from google.colab import drive

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# =============================================================================
# 2. CONFIGURATION
# =============================================================================

# UPDATE THIS PATH if your files are elsewhere
TENSOR_FOLDER = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/benchmarks/MTGNN_semantic_tensors_notop25"

# Path to original data (needed to recreate scaler)
GRAPH_FOLDER_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn"
TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_weekly_aggregated_by_week_keyword.parquet"

# Features used in training
FEATURE_COLS = [
    'impressions_sum', 'cpc_week',
    'avg_sim_top25_this_week', 'avg_sim_top25_last_week',
    'n_sim_this_week', 'n_sim_last_week',
    'adclicks_sum', 'adcost_sum',
    'n_dev_desktop', 'n_dev_mobile', 'n_dev_tablet',
    'n_st_branded_search', 'n_st_generic_search',
]
TARGET_COL = 'cpc_week'
TEST_WEEKS_LAST = 12

# =============================================================================
# 3. HELPER FUNCTIONS (Scaler & Metrics)
# =============================================================================

def get_fitted_scaler(graph_folder, time_series_path):
    """Recreates the scaler used in training."""
    print("Recreating Scaler...")

    # Load Data
    with open(os.path.join(graph_folder, 'keyword_map.json'), 'r') as f:
        keyword_map = json.load(f)
    df = pd.read_parquet(time_series_path)

    # Parse Weeks
    if df['week'].dtype == object and df['week'].astype(str).str.contains('-').any():
        parts = df['week'].astype(str).str.split('-', expand=True)
        week_nums = pd.to_numeric(parts[0], errors='coerce')
        years = pd.to_numeric(parts[1], errors='coerce')
        df['week'] = years * 100 + week_nums
    else:
        df['week'] = pd.to_numeric(df['week'], errors='coerce')

    df = df.dropna(subset=['week'])

    # Filter Training Weeks
    weeks = np.array(sorted(df['week'].unique()))
    trainval_weeks = weeks[:-TEST_WEEKS_LAST]
    val_ratio = 0.25
    split_idx = int(len(trainval_weeks) * (1 - val_ratio))
    train_weeks = trainval_weeks[:split_idx]

    print(f"  Fitting on {len(train_weeks)} training weeks...")

    # Build Matrix for Scaler
    df_train = df[df['week'].isin(train_weeks)]
    X_flat = df_train[FEATURE_COLS].values

    # Handle Log1p for Target (Index 1)
    target_idx = FEATURE_COLS.index(TARGET_COL)
    X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

    scaler = StandardScaler()
    scaler.fit(X_flat)

    print("  Scaler ready.")
    return scaler, target_idx


def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
    """
    Unscales tensors and computes per-node (keyword) metrics:
    - avg SMAPE across keywords
    - median SMAPE across keywords
    - avg RMSE across keywords
    - median RMSE across keywords
    """
    # Ensure CPU numpy
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.cpu().numpy()

    preds_flat = preds.reshape(-1, 1)
    targets_flat = targets.reshape(-1, 1)

    # Inverse Scale only on target column
    num_features = scaler.mean_.shape[0]
    dummy_preds = np.zeros((len(preds_flat), num_features))
    dummy_targets = np.zeros((len(targets_flat), num_features))

    dummy_preds[:, target_col_idx] = preds_flat[:, 0]
    dummy_targets[:, target_col_idx] = targets_flat[:, 0]

    real_preds = scaler.inverse_transform(dummy_preds)[:, target_col_idx]
    real_targets = scaler.inverse_transform(dummy_targets)[:, target_col_idx]

    # Reverse Log1p
    real_preds = np.expm1(real_preds)
    real_targets = np.expm1(real_targets)
    real_preds = np.maximum(real_preds, 0.0)

    # Try to reshape into [Time, Nodes] to get per-node metrics
    try:
        P = torch.from_numpy(real_preds.reshape(-1, 1811))  # [T, N]
        A = torch.from_numpy(real_targets.reshape(-1, 1811))  # [T, N]

        # Per-node metrics (average over time)
        node_mse = torch.mean((P - A) ** 2, dim=0)      # [N]
        node_rmse = torch.sqrt(node_mse)                # [N]

        numerator = torch.abs(P - A)
        denominator = (torch.abs(P) + torch.abs(A)) / 2.0
        node_smape = 100 * torch.mean(numerator / (denominator + 1e-8), dim=0)  # [N]

        # Average and median across keywords
        avg_rmse = torch.mean(node_rmse).item()
        med_rmse = torch.median(node_rmse).item()
        avg_smape = torch.mean(node_smape).item()
        med_smape = torch.median(node_smape).item()

    except Exception as e:
        print(f"Warning: could not reshape into [T, N] for per-node metrics: {e}")
        # Fallback: treat everything as a flat list (no per-keyword metrics)
        P = torch.from_numpy(real_preds)
        A = torch.from_numpy(real_targets)
        # Global metrics as a fallback; set node-based ones to NaN
        mse = torch.mean((P - A) ** 2).item()
        rmse = np.sqrt(mse)
        numerator = torch.abs(P - A)
        denominator = (torch.abs(P) + torch.abs(A)) / 2.0
        smape = 100 * torch.mean(numerator / (denominator + 1e-8)).item()

        avg_rmse = rmse
        med_rmse = np.nan
        avg_smape = smape
        med_smape = np.nan

    return avg_smape, med_smape, avg_rmse, med_rmse


# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================

# A. Init Scaler
scaler, target_col_idx = get_fitted_scaler(GRAPH_FOLDER_PATH, TIME_SERIES_CSV_PATH)

# B. File Discovery
print("\n" + "="*80)
print(f"SEARCHING FOR TENSORS IN: {TENSOR_FOLDER}")
print("="*80)

if not os.path.exists(TENSOR_FOLDER):
    print(f"❌ ERROR: Folder not found: {TENSOR_FOLDER}")
    print("Please check the path and try again.")
else:
    all_files = os.listdir(TENSOR_FOLDER)
    pred_files = [f for f in all_files if f.endswith('_predictions.pt')]
    print(f"Found {len(all_files)} total files.")
    print(f"Found {len(pred_files)} prediction files to process.")

    if len(pred_files) == 0:
        print("  (No '_predictions.pt' files found. Did you save them?)")
        print("  First 5 files in folder:", all_files[:5])

    results_list = []

    print(f"\n{'Horizon':<8} | {'Model':<20} | {'Avg SMAPE':<10} | {'Med SMAPE':<10} | {'Avg RMSE':<10} | {'Med RMSE':<10}")
    print("-" * 90)

    for p_file in sorted(pred_files):
        # Infer target filename
        t_file = p_file.replace('_predictions.pt', '_targets.pt')

        if t_file not in all_files:
            print(f"⚠ Missing targets for {p_file}, skipping.")
            continue

        # Parse Model/Horizon from filename
        # Expected format: "{Model}_h{Horizon}_predictions.pt"
        try:
            match = re.search(r'(.+)_h(\d+)_predictions\.pt', p_file)
            if match:
                model_name = match.group(1)
                horizon = int(match.group(2))
            else:
                model_name = p_file
                horizon = 0
        except:
            model_name = p_file
            horizon = 0

        # Load & Calculate
        try:
            p_path = os.path.join(TENSOR_FOLDER, p_file)
            t_path = os.path.join(TENSOR_FOLDER, t_file)

            preds = torch.load(p_path, map_location='cpu')
            targets = torch.load(t_path, map_location='cpu')

            avg_smape, med_smape, avg_rmse, med_rmse = calculate_advanced_metrics(
                preds, targets, scaler, target_col_idx
            )

            print(f"{horizon:<8} | {model_name:<20} | {avg_smape:<10.2f} | {med_smape:<10.2f} | {avg_rmse:<10.4f} | {med_rmse:<10.4f}")

            results_list.append({
                'Horizon': horizon,
                'Model': model_name,
                'Avg_Node_SMAPE': avg_smape,
                'Median_Node_SMAPE': med_smape,
                'Avg_Node_RMSE': avg_rmse,
                'Median_Node_RMSE': med_rmse
            })

        except Exception as e:
            print(f"Error processing {p_file}: {e}")

    # Save
    if results_list:
        df_res = pd.DataFrame(results_list).sort_values(['Horizon', 'Avg_Node_SMAPE'])
        df_res.to_csv("final_detailed_metrics.csv", index=False)
        print("\n✓ Saved metrics to final_detailed_metrics.csv")

Recreating Scaler...
  Fitting on 86 training weeks...
  Scaler ready.

SEARCHING FOR TENSORS IN: /content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/benchmarks/MTGNN_semantic_tensors_notop25
Found 12 total files.
Found 3 prediction files to process.

Horizon  | Model                | Avg SMAPE  | Med SMAPE  | Avg RMSE   | Med RMSE  
------------------------------------------------------------------------------------------
12       | MTGNN                | 37.89      | 28.26      | 1.0630     | 0.6714    
1        | MTGNN                | 27.41      | 23.02      | 0.9700     | 0.7415    
6        | MTGNN                | 32.90      | 28.13      | 1.1035     | 0.8537    

✓ Saved metrics to final_detailed_metrics.csv


# with std

In [ ]:
  # =============================================================================
  # 1. SETUP & IMPORTS
  # =============================================================================
  import os
  import json
  import torch
  import numpy as np
  import pandas as pd
  import re
  from sklearn.preprocessing import StandardScaler
  from google.colab import drive

  # Mount Drive
  if not os.path.exists('/content/drive'):
      drive.mount('/content/drive')

  # =============================================================================
  # 2. CONFIGURATION
  # =============================================================================

  # UPDATE THIS PATH if your files are elsewhere
  TENSOR_FOLDER = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/exp2_tensors"

  # Path to original data (needed to recreate scaler)
  GRAPH_FOLDER_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn"
  TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/final_forecast_ready.parquet"

  # Features used in training
  FEATURE_COLS = [

      # === Core Operational Metrics ===
      'impressions_sum',
      'cpc_week',

      # === Semantic Similarity Features ===
      'avg_sim_top25_this_week',
      'avg_sim_top25_last_week',
      'n_sim_this_week',
      'n_sim_last_week',

      # === Domain Share (Competitors) ===
      'dom_share_avis',
      'dom_share_avisautonoleggio',
      'dom_share_aviscarsales',
      'dom_share_budget',
      'dom_share_budgetautonoleggio',
      'dom_share_dollar',
      'dom_share_economybookings',
      'dom_share_hertz',
      'dom_share_letsdrive',
      'dom_share_sixt',
      'dom_share_thrifty',

      # === Device Distribution ===
      'n_dev_desktop',
      'n_dev_mobile',
      'n_dev_tablet',

      # === Search Type Distribution ===
      'n_st_branded_search',
      'n_st_generic_search',

      # === DTW Neighbour Features (Graph-based) ===
      'dtw_neighbour_1',
      'dtw_neighbour_2',
      'dtw_neighbour_3',
      'dtw_neighbour_4',
      'dtw_neighbour_5',
      'dtw_neighbour_6',
      'dtw_neighbour_7',
      'dtw_neighbour_8',
      'dtw_neighbour_9',
      'dtw_neighbour_10',
      'dtw_neighbour_11',
      'dtw_neighbour_12',
      'dtw_neighbour_13',
      'dtw_neighbour_14',
      'dtw_neighbour_15',
      'dtw_neighbour_16',
      'dtw_neighbour_17',
      'dtw_neighbour_18',
      'dtw_neighbour_19',
      'dtw_neighbour_20',

      # # === Geographic Features ===
      # 'detected_city',
      # 'detected_country',
      # 'detected_continent',
      'detected_city_id',
      'detected_country_id',
      'detected_continent_id',

      # === Autoregressive / Time-Series Features ===
      'target_lag_1',
      'target_lag_2',
      'target_lag_4',
      'target_lag_12',
      'target_roll_mean_4_lag1',

      # === Semantic PCA Components ===
      'sem_pc_0',
      'sem_pc_1',
      'sem_pc_2',
      'sem_pc_4',
  ]
  TARGET_COL = 'cpc_week'
  TEST_WEEKS_LAST = 12

  # =============================================================================
  # 3. HELPER FUNCTIONS (Scaler & Metrics)
  # =============================================================================

  def get_fitted_scaler(graph_folder, time_series_path):
      """Recreates the scaler used in training."""
      print("Recreating Scaler...")

      # Load Data
      with open(os.path.join(graph_folder, 'keyword_map.json'), 'r') as f:
          keyword_map = json.load(f)
      df = pd.read_parquet(time_series_path)

      # --- MUST MATCH TRAINING: Geo-Categorical Conversion ---
      df["detected_city"] = df["detected_city"].astype(str).replace('None', 'Unknown').fillna('Unknown')
      df["detected_country"] = df["detected_country"].astype(str).replace('None', 'Unknown').fillna('Unknown')
      df["detected_continent"] = df["detected_continent"].astype(str).replace('None', 'Unknown').fillna('Unknown')

      city_to_id = {name: i for i, name in enumerate(sorted(df["detected_city"].unique()), start=1)}
      country_to_id = {name: i for i, name in enumerate(sorted(df["detected_country"].unique()), start=1)}
      continent_to_id = {name: i for i, name in enumerate(sorted(df["detected_continent"].unique()), start=1)}

      df["detected_city_id"] = df["detected_city"].map(city_to_id).astype("int32")
      df["detected_country_id"] = df["detected_country"].map(country_to_id).astype("int32")
      df["detected_continent_id"] = df["detected_continent"].map(continent_to_id).astype("int32")
      # ------------------------------------------------------

      # Parse Weeks
      if df['week'].dtype == object and df['week'].astype(str).str.contains('-').any():
          parts = df['week'].astype(str).str.split('-', expand=True)
          week_nums = pd.to_numeric(parts[0], errors='coerce')
          years = pd.to_numeric(parts[1], errors='coerce')
          df['week'] = years * 100 + week_nums
      else:
          df['week'] = pd.to_numeric(df['week'], errors='coerce')

      df = df.dropna(subset=['week'])

      # Filter Training Weeks
      weeks = np.array(sorted(df['week'].unique()))
      trainval_weeks = weeks[:-TEST_WEEKS_LAST]
      val_ratio = 0.25
      split_idx = int(len(trainval_weeks) * (1 - val_ratio))
      train_weeks = trainval_weeks[:split_idx]

      print(f"  Fitting on {len(train_weeks)} training weeks...")

      # Build Matrix for Scaler
      df_train = df[df['week'].isin(train_weeks)]
      X_flat = df_train[FEATURE_COLS].values

      # Handle Log1p for Target (Index 1)
      target_idx = FEATURE_COLS.index(TARGET_COL)
      X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

      scaler = StandardScaler()
      scaler.fit(X_flat)

      print("  Scaler ready.")
      return scaler, target_idx


  def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
      """
      Unscales tensors and computes per-node (keyword) metrics:
      - avg SMAPE across keywords
      - median SMAPE across keywords
      - std SMAPE across keywords
      - avg RMSE across keywords
      - median RMSE across keywords
      - std RMSE across keywords
      """
      # Ensure CPU numpy
      if isinstance(preds, torch.Tensor):
          preds = preds.cpu().numpy()
      if isinstance(targets, torch.Tensor):
          targets = targets.cpu().numpy()

      preds_flat = preds.reshape(-1, 1)
      targets_flat = targets.reshape(-1, 1)

      # Inverse Scale only on target column
      num_features = scaler.mean_.shape[0]
      dummy_preds = np.zeros((len(preds_flat), num_features))
      dummy_targets = np.zeros((len(targets_flat), num_features))

      dummy_preds[:, target_col_idx] = preds_flat[:, 0]
      dummy_targets[:, target_col_idx] = targets_flat[:, 0]

      real_preds = scaler.inverse_transform(dummy_preds)[:, target_col_idx]
      real_targets = scaler.inverse_transform(dummy_targets)[:, target_col_idx]

      # Reverse Log1p
      real_preds = np.expm1(real_preds)
      real_targets = np.expm1(real_targets)
      real_preds = np.maximum(real_preds, 0.0)

      # Try to reshape into [Time, Nodes] to get per-node metrics
      try:
          P = torch.from_numpy(real_preds.reshape(-1, 1811))  # [T, N]
          A = torch.from_numpy(real_targets.reshape(-1, 1811))  # [T, N]

          # Per-node metrics (average over time)
          node_mse = torch.mean((P - A) ** 2, dim=0)      # [N]
          node_rmse = torch.sqrt(node_mse)                # [N]

          numerator = torch.abs(P - A)
          denominator = (torch.abs(P) + torch.abs(A)) / 2.0
          node_smape = 100 * torch.mean(numerator / (denominator + 1e-8), dim=0)  # [N]

          # Average, median, and standard deviation across keywords
          avg_rmse = torch.mean(node_rmse).item()
          med_rmse = torch.median(node_rmse).item()
          std_rmse = torch.std(node_rmse).item()

          avg_smape = torch.mean(node_smape).item()
          med_smape = torch.median(node_smape).item()
          std_smape = torch.std(node_smape).item()

      except Exception as e:
          print(f"Warning: could not reshape into [T, N] for per-node metrics: {e}")
          # Fallback: treat everything as a flat list (no per-keyword metrics)
          P = torch.from_numpy(real_preds)
          A = torch.from_numpy(real_targets)
          # Global metrics as a fallback; set node-based ones to NaN
          mse = torch.mean((P - A) ** 2).item()
          rmse = np.sqrt(mse)
          numerator = torch.abs(P - A)
          denominator = (torch.abs(P) + torch.abs(A)) / 2.0
          smape = 100 * torch.mean(numerator / (denominator + 1e-8)).item()

          avg_rmse = rmse
          med_rmse = np.nan
          std_rmse = np.nan
          avg_smape = smape
          med_smape = np.nan
          std_smape = np.nan

      return avg_smape, med_smape, std_smape, avg_rmse, med_rmse, std_rmse


  # =============================================================================
  # 4. MAIN EXECUTION
  # =============================================================================

  # A. Init Scaler
  scaler, target_col_idx = get_fitted_scaler(GRAPH_FOLDER_PATH, TIME_SERIES_CSV_PATH)

  # B. File Discovery
  print("\n" + "="*80)
  print(f"SEARCHING FOR TENSORS IN: {TENSOR_FOLDER}")
  print("="*80)

  if not os.path.exists(TENSOR_FOLDER):
      print(f"❌ ERROR: Folder not found: {TENSOR_FOLDER}")
      print("Please check the path and try again.")
  else:
      all_files = os.listdir(TENSOR_FOLDER)
      pred_files = [f for f in all_files if f.endswith('_predictions.pt')]
      print(f"Found {len(all_files)} total files.")
      print(f"Found {len(pred_files)} prediction files to process.")

      if len(pred_files) == 0:
          print("  (No '_predictions.pt' files found. Did you save them?)")
          print("  First 5 files in folder:", all_files[:5])

      results_list = []

      print(f"\n{'Horizon':<8} | {'Model':<20} | {'Avg SMAPE':<10} | {'Med SMAPE':<10} | {'Std SMAPE':<10} | {'Avg RMSE':<10} | {'Med RMSE':<10} | {'Std RMSE':<10}")
      print("-" * 110)

      for p_file in sorted(pred_files):
          # Infer target filename
          t_file = p_file.replace('_predictions.pt', '_targets.pt')

          if t_file not in all_files:
              print(f"⚠ Missing targets for {p_file}, skipping.")
              continue

          # Parse Model/Horizon from filename
          # Expected format: "{Model}_h{Horizon}_predictions.pt"
          try:
              match = re.search(r'(.+)_h(\d+)_predictions\.pt', p_file)
              if match:
                  model_name = match.group(1)
                  horizon = int(match.group(2))
              else:
                  model_name = p_file
                  horizon = 0
          except:
              model_name = p_file
              horizon = 0

          # Load & Calculate
          try:
              p_path = os.path.join(TENSOR_FOLDER, p_file)
              t_path = os.path.join(TENSOR_FOLDER, t_file)

              preds = torch.load(p_path, map_location='cpu')
              targets = torch.load(t_path, map_location='cpu')

              avg_smape, med_smape, std_smape, avg_rmse, med_rmse, std_rmse = calculate_advanced_metrics(
                  preds, targets, scaler, target_col_idx
              )

              print(f"{horizon:<8} | {model_name:<20} | {avg_smape:<10.2f} | {med_smape:<10.2f} | {std_smape:<10.2f} | {avg_rmse:<10.4f} | {med_rmse:<10.4f} | {std_rmse:<10.4f}")

              results_list.append({
                  'Horizon': horizon,
                  'Model': model_name,
                  'Avg_Node_SMAPE': avg_smape,
                  'Median_Node_SMAPE': med_smape,
                  'Std_Node_SMAPE': std_smape,
                  'Avg_Node_RMSE': avg_rmse,
                  'Median_Node_RMSE': med_rmse,
                  'Std_Node_RMSE': std_rmse
              })

          except Exception as e:
              print(f"Error processing {p_file}: {e}")

      # Save
      if results_list:
          df_res = pd.DataFrame(results_list).sort_values(['Horizon', 'Avg_Node_SMAPE'])
          df_res.to_csv("final_detailed_metrics.csv", index=False)
          print("\n✓ Saved metrics to final_detailed_metrics.csv")

Recreating Scaler...
  Fitting on 85 training weeks...
  Scaler ready.

SEARCHING FOR TENSORS IN: /content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/exp2_tensors
Found 96 total files.
Found 24 prediction files to process.

Horizon  | Model                | Avg SMAPE  | Med SMAPE  | Std SMAPE  | Avg RMSE   | Med RMSE   | Std RMSE  
--------------------------------------------------------------------------------------------------------------
12       | A3TGCN               | 48.06      | 38.89      | 39.24      | 1.3304     | 0.9406     | 1.4557    
1        | A3TGCN               | 38.82      | 33.86      | 20.53      | 1.2521     | 1.0194     | 0.9803    
6        | A3TGCN               | 44.56      | 39.75      | 24.55      | 1.3954     | 1.0168     | 1.3165    
12       | AGCRN                | 39.35      | 30.36      | 36.93      | 1.0579     | 0.7020     | 1.2348    
1        | AGCRN                | 28.10      | 23.63      | 17.37      | 0.9548     | 0.7436     | 

# new

In [ ]:
# =============================================================================
# 1. SETUP & IMPORTS
# =============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
from google.colab import drive

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# =============================================================================
# 2. CONFIGURATION
# =============================================================================

# Folder containing ALL prediction/target tensors for all k values
TENSOR_FOLDER = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/k_experiment_tensors_notop25"

# Base path to multiple k-graphs (each k has its own subfolder: k5, k10, ...)
GRAPH_BASE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn_multiple"

# Path to original data (needed to recreate scaler)
TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_weekly_aggregated_by_week_keyword.parquet"

# Features used in training
FEATURE_COLS = [
    'impressions_sum', 'cpc_week',
    # 'avg_sim_top25_this_week', 'avg_sim_top25_last_week',
    # 'n_sim_this_week', 'n_sim_last_week',
    'adclicks_sum', 'adcost_sum',
    'n_dev_desktop', 'n_dev_mobile', 'n_dev_tablet',
    'n_st_branded_search', 'n_st_generic_search',
]
TARGET_COL = 'cpc_week'
TEST_WEEKS_LAST = 12

# If you want, you can expose this as a config instead of hard-coding
N_NODES = 1811

# =============================================================================
# 3. HELPER FUNCTIONS (Scaler & Metrics)
# =============================================================================

def get_fitted_scaler(graph_folder, time_series_path):
    """Recreates the scaler used in training for a given k-graph folder."""
    print(f"Recreating Scaler for graph folder: {graph_folder}")

    # Load Data
    keyword_map_path = os.path.join(graph_folder, 'keyword_map.json')
    with open(keyword_map_path, 'r') as f:
        keyword_map = json.load(f)
    # keyword_map is currently not used explicitly, but ensures consistency.

    df = pd.read_parquet(time_series_path)

    # Parse Weeks
    if df['week'].dtype == object and df['week'].astype(str).str.contains('-').any():
        parts = df['week'].astype(str).str.split('-', expand=True)
        week_nums = pd.to_numeric(parts[0], errors='coerce')
        years = pd.to_numeric(parts[1], errors='coerce')
        df['week'] = years * 100 + week_nums
    else:
        df['week'] = pd.to_numeric(df['week'], errors='coerce')

    df = df.dropna(subset=['week'])

    # Filter Training Weeks
    weeks = np.array(sorted(df['week'].unique()))
    trainval_weeks = weeks[:-TEST_WEEKS_LAST]
    val_ratio = 0.25
    split_idx = int(len(trainval_weeks) * (1 - val_ratio))
    train_weeks = trainval_weeks[:split_idx]

    print(f"  Fitting on {len(train_weeks)} training weeks...")

    # Build Matrix for Scaler
    df_train = df[df['week'].isin(train_weeks)]
    X_flat = df_train[FEATURE_COLS].values

    # Handle Log1p for Target
    target_idx = FEATURE_COLS.index(TARGET_COL)
    X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

    scaler = StandardScaler()
    scaler.fit(X_flat)

    print("  Scaler ready.")
    return scaler, target_idx


def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
    """
    Unscales tensors and computes per-node (keyword) metrics:
    - avg SMAPE across keywords
    - median SMAPE across keywords
    - avg RMSE across keywords
    - median RMSE across keywords
    """
    # Ensure CPU numpy
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.cpu().numpy()

    preds_flat = preds.reshape(-1, 1)
    targets_flat = targets.reshape(-1, 1)

    # Inverse Scale only on target column
    num_features = scaler.mean_.shape[0]
    dummy_preds = np.zeros((len(preds_flat), num_features))
    dummy_targets = np.zeros((len(targets_flat), num_features))

    dummy_preds[:, target_col_idx] = preds_flat[:, 0]
    dummy_targets[:, target_col_idx] = targets_flat[:, 0]

    real_preds = scaler.inverse_transform(dummy_preds)[:, target_col_idx]
    real_targets = scaler.inverse_transform(dummy_targets)[:, target_col_idx]

    # Reverse Log1p
    real_preds = np.expm1(real_preds)
    real_targets = np.expm1(real_targets)
    real_preds = np.maximum(real_preds, 0.0)

    # Try to reshape into [Time, Nodes] to get per-node metrics
    try:
        P = torch.from_numpy(real_preds.reshape(-1, N_NODES))  # [T, N]
        A = torch.from_numpy(real_targets.reshape(-1, N_NODES))  # [T, N]

        # Per-node metrics (average over time)
        node_mse = torch.mean((P - A) ** 2, dim=0)      # [N]
        node_rmse = torch.sqrt(node_mse)                # [N]

        numerator = torch.abs(P - A)
        denominator = (torch.abs(P) + torch.abs(A)) / 2.0
        node_smape = 100 * torch.mean(numerator / (denominator + 1e-8), dim=0)  # [N]

        # Average and median across keywords
        avg_rmse = torch.mean(node_rmse).item()
        med_rmse = torch.median(node_rmse).item()
        avg_smape = torch.mean(node_smape).item()
        med_smape = torch.median(node_smape).item()

    except Exception as e:
        print(f"Warning: could not reshape into [T, N] for per-node metrics: {e}")
        # Fallback: treat everything as a flat list (no per-keyword metrics)
        P = torch.from_numpy(real_preds)
        A = torch.from_numpy(real_targets)
        # Global metrics as a fallback; set node-based ones to NaN
        mse = torch.mean((P - A) ** 2).item()
        rmse = np.sqrt(mse)
        numerator = torch.abs(P - A)
        denominator = (torch.abs(P) + torch.abs(A)) / 2.0
        smape = 100 * torch.mean(numerator / (denominator + 1e-8)).item()

        avg_rmse = rmse
        med_rmse = np.nan
        avg_smape = smape
        med_smape = np.nan

    return avg_smape, med_smape, avg_rmse, med_rmse


# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================

# We now create a scaler per k value on demand, and cache it.
scalers_by_k = {}  # k -> (scaler, target_col_idx)

print("\n" + "="*80)
print(f"SEARCHING FOR TENSORS IN: {TENSOR_FOLDER}")
print("="*80)

if not os.path.exists(TENSOR_FOLDER):
    print(f"❌ ERROR: Folder not found: {TENSOR_FOLDER}")
    print("Please check the path and try again.")
else:
    all_files = os.listdir(TENSOR_FOLDER)
    pred_files = [f for f in all_files if f.endswith('_predictions.pt')]
    print(f"Found {len(all_files)} total files.")
    print(f"Found {len(pred_files)} prediction files to process.")

    if len(pred_files) == 0:
        print("  (No '_predictions.pt' files found. Did you save them?)")
        print("  First 5 files in folder:", all_files[:5])

    results_list = []

    print(f"\n{'k':<4} | {'Horizon':<8} | {'Model':<20} | {'Avg SMAPE':<10} | {'Med SMAPE':<10} | {'Avg RMSE':<10} | {'Med RMSE':<10}")
    print("-" * 100)

    for p_file in sorted(pred_files):
        # Infer target filename
        t_file = p_file.replace('_predictions.pt', '_targets.pt')

        if t_file not in all_files:
            print(f"⚠ Missing targets for {p_file}, skipping.")
            continue

        # Parse k, Model, Horizon from filename
        # Expected new format: "k5_A3TGCN_h6_predictions.pt"
        k_val = None
        model_name = None
        horizon = 0

        m_new = re.search(r'k(\d+)_(.+)_h(\d+)_predictions\.pt', p_file)
        if m_new:
            k_val = int(m_new.group(1))
            model_name = m_new.group(2)
            horizon = int(m_new.group(3))
        else:
            # Fallback to old format: "{Model}_h{Horizon}_predictions.pt"
            m_old = re.search(r'(.+)_h(\d+)_predictions\.pt', p_file)
            if m_old:
                model_name = m_old.group(1)
                horizon = int(m_old.group(2))
            else:
                model_name = p_file
                horizon = 0

        # Ensure we have a scaler for this k (if k encoded)
        if k_val is not None:
            if k_val not in scalers_by_k:
                graph_folder = os.path.join(GRAPH_BASE_FOLDER, f'k{k_val}')
                if not os.path.exists(graph_folder):
                    print(f"❌ Graph folder not found for k={k_val}: {graph_folder}")
                    print("   Skipping all files with this k.")
                    continue
                scaler_k, target_idx_k = get_fitted_scaler(graph_folder, TIME_SERIES_CSV_PATH)
                scalers_by_k[k_val] = (scaler_k, target_idx_k)
            scaler, target_col_idx = scalers_by_k[k_val]
        else:
            # If no k in filename, you could either skip or use a default scaler.
            # Here we just create/use a generic scaler once (k=None).
            if None not in scalers_by_k:
                # Use base graph folder (e.g., k5 or old single-graph path)
                default_graph_folder = os.path.join(GRAPH_BASE_FOLDER, "k5")
                scaler_default, target_idx_default = get_fitted_scaler(default_graph_folder, TIME_SERIES_CSV_PATH)
                scalers_by_k[None] = (scaler_default, target_idx_default)
            scaler, target_col_idx = scalers_by_k[None]

        # Load & Calculate
        try:
            p_path = os.path.join(TENSOR_FOLDER, p_file)
            t_path = os.path.join(TENSOR_FOLDER, t_file)

            preds = torch.load(p_path, map_location='cpu')
            targets = torch.load(t_path, map_location='cpu')

            avg_smape, med_smape, avg_rmse, med_rmse = calculate_advanced_metrics(
                preds, targets, scaler, target_col_idx
            )

            k_str = str(k_val) if k_val is not None else "-"
            print(f"{k_str:<4} | {horizon:<8} | {model_name:<20} | {avg_smape:<10.2f} | {med_smape:<10.2f} | {avg_rmse:<10.4f} | {med_rmse:<10.4f}")

            results_list.append({
                'k': k_val,
                'Horizon': horizon,
                'Model': model_name,
                'Avg_Node_SMAPE': avg_smape,
                'Median_Node_SMAPE': med_smape,
                'Avg_Node_RMSE': avg_rmse,
                'Median_Node_RMSE': med_rmse
            })

        except Exception as e:
            print(f"Error processing {p_file}: {e}")

    # Save
    if results_list:
        df_res = pd.DataFrame(results_list).sort_values(['k', 'Horizon', 'Avg_Node_SMAPE'])
        df_res.to_csv("final_detailed_metrics_multi_k.csv", index=False)
        print("\n✓ Saved metrics to final_detailed_metrics_multi_k.csv")


SEARCHING FOR TENSORS IN: /content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/k_experiment_tensors_notop25
Found 72 total files.
Found 24 prediction files to process.

k    | Horizon  | Model                | Avg SMAPE  | Med SMAPE  | Avg RMSE   | Med RMSE  
----------------------------------------------------------------------------------------------------
Recreating Scaler for graph folder: /content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn_multiple/k10
  Fitting on 86 training weeks...
  Scaler ready.
10   | 6        | A3TGCN               | 40.22      | 34.37      | 1.3234     | 0.9773    
10   | 6        | DCRNN                | 31.66      | 26.76      | 1.0731     | 0.7830    
10   | 6        | GConvLSTM            | 33.26      | 27.70      | 1.1349     | 0.8119    
10   | 6        | GraphWaveNet         | 30.86      | 25.31      | 1.0613     | 0.7609    
10   | 6        | STConv               | 40.17      | 33.97      | 1.3336     | 0.924

# with std

In [ ]:
# =============================================================================
# 1. SETUP & IMPORTS
# =============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
from google.colab import drive

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# =============================================================================
# 2. CONFIGURATION
# =============================================================================

# Folder containing ALL prediction/target tensors for all k values
TENSOR_FOLDER = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/thresh_tensors_weighted_notop25"

# Base path to multiple k-graphs (each k has its own subfolder: k5, k10, ...)
GRAPH_BASE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn_multiple"

# Path to original data (needed to recreate scaler)
TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_weekly_aggregated_by_week_keyword.parquet"

# Features used in training
FEATURE_COLS = [
    'impressions_sum', 'cpc_week',
    # 'avg_sim_top25_this_week', 'avg_sim_top25_last_week',
    # 'n_sim_this_week', 'n_sim_last_week',
    'adclicks_sum', 'adcost_sum',
    'n_dev_desktop', 'n_dev_mobile', 'n_dev_tablet',
    'n_st_branded_search', 'n_st_generic_search',
]
TARGET_COL = 'cpc_week'
TEST_WEEKS_LAST = 12

# If you want, you can expose this as a config instead of hard-coding
N_NODES = 1811

# =============================================================================
# 3. HELPER FUNCTIONS (Scaler & Metrics)
# =============================================================================

def get_fitted_scaler(graph_folder, time_series_path):
    """Recreates the scaler used in training for a given k-graph folder."""
    print(f"Recreating Scaler for graph folder: {graph_folder}")

    # Load Data
    keyword_map_path = os.path.join(graph_folder, 'keyword_map.json')
    with open(keyword_map_path, 'r') as f:
        keyword_map = json.load(f)
    # keyword_map is currently not used explicitly, but ensures consistency.

    df = pd.read_parquet(time_series_path)

    # Parse Weeks
    if df['week'].dtype == object and df['week'].astype(str).str.contains('-').any():
        parts = df['week'].astype(str).str.split('-', expand=True)
        week_nums = pd.to_numeric(parts[0], errors='coerce')
        years = pd.to_numeric(parts[1], errors='coerce')
        df['week'] = years * 100 + week_nums
    else:
        df['week'] = pd.to_numeric(df['week'], errors='coerce')

    df = df.dropna(subset=['week'])

    # Filter Training Weeks
    weeks = np.array(sorted(df['week'].unique()))
    trainval_weeks = weeks[:-TEST_WEEKS_LAST]
    val_ratio = 0.25
    split_idx = int(len(trainval_weeks) * (1 - val_ratio))
    train_weeks = trainval_weeks[:split_idx]

    print(f"  Fitting on {len(train_weeks)} training weeks...")

    # Build Matrix for Scaler
    df_train = df[df['week'].isin(train_weeks)]
    X_flat = df_train[FEATURE_COLS].values

    # Handle Log1p for Target
    target_idx = FEATURE_COLS.index(TARGET_COL)
    X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

    scaler = StandardScaler()
    scaler.fit(X_flat)

    print("  Scaler ready.")
    return scaler, target_idx


def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
    """
    Unscales tensors and computes per-node (keyword) metrics:
    - avg SMAPE across keywords
    - median SMAPE across keywords
    - std SMAPE across keywords
    - avg RMSE across keywords
    - median RMSE across keywords
    - std RMSE across keywords
    """
    # Ensure CPU numpy
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.cpu().numpy()

    preds_flat = preds.reshape(-1, 1)
    targets_flat = targets.reshape(-1, 1)

    # Inverse Scale only on target column
    num_features = scaler.mean_.shape[0]
    dummy_preds = np.zeros((len(preds_flat), num_features))
    dummy_targets = np.zeros((len(targets_flat), num_features))

    dummy_preds[:, target_col_idx] = preds_flat[:, 0]
    dummy_targets[:, target_col_idx] = targets_flat[:, 0]

    real_preds = scaler.inverse_transform(dummy_preds)[:, target_col_idx]
    real_targets = scaler.inverse_transform(dummy_targets)[:, target_col_idx]

    # Reverse Log1p
    real_preds = np.expm1(real_preds)
    real_targets = np.expm1(real_targets)
    real_preds = np.maximum(real_preds, 0.0)

    # Try to reshape into [Time, Nodes] to get per-node metrics
    try:
        P = torch.from_numpy(real_preds.reshape(-1, N_NODES))  # [T, N]
        A = torch.from_numpy(real_targets.reshape(-1, N_NODES))  # [T, N]

        # Per-node metrics (average over time)
        node_mse = torch.mean((P - A) ** 2, dim=0)      # [N]
        node_rmse = torch.sqrt(node_mse)                # [N]

        numerator = torch.abs(P - A)
        denominator = (torch.abs(P) + torch.abs(A)) / 2.0
        node_smape = 100 * torch.mean(numerator / (denominator + 1e-8), dim=0)  # [N]

        # Average, median, and standard deviation across keywords
        avg_rmse = torch.mean(node_rmse).item()
        med_rmse = torch.median(node_rmse).item()
        std_rmse = torch.std(node_rmse).item()

        avg_smape = torch.mean(node_smape).item()
        med_smape = torch.median(node_smape).item()
        std_smape = torch.std(node_smape).item()

    except Exception as e:
        print(f"Warning: could not reshape into [T, N] for per-node metrics: {e}")
        # Fallback: treat everything as a flat list (no per-keyword metrics)
        P = torch.from_numpy(real_preds)
        A = torch.from_numpy(real_targets)
        # Global metrics as a fallback; set node-based ones to NaN
        mse = torch.mean((P - A) ** 2).item()
        rmse = np.sqrt(mse)
        numerator = torch.abs(P - A)
        denominator = (torch.abs(P) + torch.abs(A)) / 2.0
        smape = 100 * torch.mean(numerator / (denominator + 1e-8)).item()

        avg_rmse = rmse
        med_rmse = np.nan
        std_rmse = np.nan
        avg_smape = smape
        med_smape = np.nan
        std_smape = np.nan

    return avg_smape, med_smape, std_smape, avg_rmse, med_rmse, std_rmse


# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================

# We now create a scaler per k value on demand, and cache it.
scalers_by_k = {}  # k -> (scaler, target_col_idx)

print("\n" + "="*80)
print(f"SEARCHING FOR TENSORS IN: {TENSOR_FOLDER}")
print("="*80)

if not os.path.exists(TENSOR_FOLDER):
    print(f"❌ ERROR: Folder not found: {TENSOR_FOLDER}")
    print("Please check the path and try again.")
else:
    all_files = os.listdir(TENSOR_FOLDER)
    pred_files = [f for f in all_files if f.endswith('_predictions.pt')]
    print(f"Found {len(all_files)} total files.")
    print(f"Found {len(pred_files)} prediction files to process.")

    if len(pred_files) == 0:
        print("  (No '_predictions.pt' files found. Did you save them?)")
        print("  First 5 files in folder:", all_files[:5])

    results_list = []

    print(f"\n{'k':<4} | {'Horizon':<8} | {'Model':<20} | {'Avg SMAPE':<10} | {'Med SMAPE':<10} | {'Std SMAPE':<10} | {'Avg RMSE':<10} | {'Med RMSE':<10} | {'Std RMSE':<10}")
    print("-" * 120)

    for p_file in sorted(pred_files):
        # Infer target filename
        t_file = p_file.replace('_predictions.pt', '_targets.pt')

        if t_file not in all_files:
            print(f"⚠ Missing targets for {p_file}, skipping.")
            continue

        # Parse k, Model, Horizon from filename
        # Expected new format: "k5_A3TGCN_h6_predictions.pt"
        k_val = None
        model_name = None
        horizon = 0

        m_new = re.search(r'k(\d+)_(.+)_h(\d+)_predictions\.pt', p_file)
        if m_new:
            k_val = int(m_new.group(1))
            model_name = m_new.group(2)
            horizon = int(m_new.group(3))
        else:
            # Fallback to old format: "{Model}_h{Horizon}_predictions.pt"
            m_old = re.search(r'(.+)_h(\d+)_predictions\.pt', p_file)
            if m_old:
                model_name = m_old.group(1)
                horizon = int(m_old.group(2))
            else:
                model_name = p_file
                horizon = 0

        # Ensure we have a scaler for this k (if k encoded)
        if k_val is not None:
            if k_val not in scalers_by_k:
                graph_folder = os.path.join(GRAPH_BASE_FOLDER, f'k{k_val}')
                if not os.path.exists(graph_folder):
                    print(f"❌ Graph folder not found for k={k_val}: {graph_folder}")
                    print("   Skipping all files with this k.")
                    continue
                scaler_k, target_idx_k = get_fitted_scaler(graph_folder, TIME_SERIES_CSV_PATH)
                scalers_by_k[k_val] = (scaler_k, target_idx_k)
            scaler, target_col_idx = scalers_by_k[k_val]
        else:
            # If no k in filename, you could either skip or use a default scaler.
            # Here we just create/use a generic scaler once (k=None).
            if None not in scalers_by_k:
                # Use base graph folder (e.g., k5 or old single-graph path)
                default_graph_folder = os.path.join(GRAPH_BASE_FOLDER, "k5")
                scaler_default, target_idx_default = get_fitted_scaler(default_graph_folder, TIME_SERIES_CSV_PATH)
                scalers_by_k[None] = (scaler_default, target_idx_default)
            scaler, target_col_idx = scalers_by_k[None]

        # Load & Calculate
        try:
            p_path = os.path.join(TENSOR_FOLDER, p_file)
            t_path = os.path.join(TENSOR_FOLDER, t_file)

            preds = torch.load(p_path, map_location='cpu')
            targets = torch.load(t_path, map_location='cpu')

            avg_smape, med_smape, std_smape, avg_rmse, med_rmse, std_rmse = calculate_advanced_metrics(
                preds, targets, scaler, target_col_idx
            )

            k_str = str(k_val) if k_val is not None else "-"
            print(f"{k_str:<4} | {horizon:<8} | {model_name:<20} | {avg_smape:<10.2f} | {med_smape:<10.2f} | {std_smape:<10.2f} | {avg_rmse:<10.4f} | {med_rmse:<10.4f} | {std_rmse:<10.4f}")

            results_list.append({
                'k': k_val,
                'Horizon': horizon,
                'Model': model_name,
                'Avg_Node_SMAPE': avg_smape,
                'Median_Node_SMAPE': med_smape,
                'Std_Node_SMAPE': std_smape,
                'Avg_Node_RMSE': avg_rmse,
                'Median_Node_RMSE': med_rmse,
                'Std_Node_RMSE': std_rmse
            })

        except Exception as e:
            print(f"Error processing {p_file}: {e}")

    # Save
    if results_list:
        df_res = pd.DataFrame(results_list).sort_values(['k', 'Horizon', 'Avg_Node_SMAPE'])
        df_res.to_csv("final_detailed_metrics_multi_k.csv", index=False)
        print("\n✓ Saved metrics to final_detailed_metrics_multi_k.csv")


SEARCHING FOR TENSORS IN: /content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/thresh_tensors_weighted_notop25
Found 51 total files.
Found 17 prediction files to process.

k    | Horizon  | Model                | Avg SMAPE  | Med SMAPE  | Std SMAPE  | Avg RMSE   | Med RMSE   | Std RMSE  
------------------------------------------------------------------------------------------------------------------------
Recreating Scaler for graph folder: /content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn_multiple/k5
  Fitting on 86 training weeks...
  Scaler ready.
-    | 6        | thresh0.6_A3TGCN     | 49.49      | 42.91      | 29.80      | 1.5856     | 1.2567     | 1.4266    
-    | 6        | thresh0.6_DCRNN      | 31.26      | 26.43      | 20.61      | 1.0701     | 0.7592     | 1.0654    
-    | 6        | thresh0.6_GConvLSTM  | 34.38      | 28.32      | 22.47      | 1.1287     | 0.8282     | 1.1220    
-    | 6        | thresh0.6_STConv     | 35.67    

# ablation study

In [ ]:
# =============================================================================
# Ablation Study: Tensors to Detailed Results
# =============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import re
from typing import Dict, List, Tuple, Any
from sklearn.preprocessing import StandardScaler

#mount google drive
from google.colab import drive
drive.mount('/content/drive')

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# ROOT directory containing the exp_XXX folders
TENSOR_ROOT = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/tensors"

# Data paths to recreate scalers
GRAPH_FOLDER_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn"
TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/final_forecast_ready.parquet"

TARGET_COL = 'cpc_week'
TEST_WEEKS_LAST = 12
VAL_RATIO = 0.25
NUM_NODES = 1811

# =============================================================================
# Feature Groups (MUST MATCH ABLATION NOTEBOOK)
# =============================================================================

CORE_FEAT = ['impressions_sum', 'cpc_week']
GEO_FEAT  = ['detected_city_id', 'detected_country_id', 'detected_continent_id']
DTW_FEAT  = [f'dtw_neighbour_{i}' for i in range(1, 21)]
LAG_FEAT  = ['target_lag_1', 'target_lag_2', 'target_lag_4', 'target_lag_12', 'target_roll_mean_4_lag1']

SIM25_FEAT = ['avg_sim_top25_this_week', 'avg_sim_top25_last_week', 'n_sim_this_week', 'n_sim_last_week']

DOM_FEAT = [
    'dom_share_avis', 'dom_share_avisautonoleggio', 'dom_share_aviscarsales', 'dom_share_budget',
    'dom_share_budgetautonoleggio', 'dom_share_dollar', 'dom_share_economybookings', 'dom_share_hertz',
    'dom_share_letsdrive', 'dom_share_sixt', 'dom_share_thrifty',
]

SEARCH_FEAT = ['n_dev_desktop', 'n_dev_mobile', 'n_dev_tablet',
    'n_st_branded_search', 'n_st_generic_search']

SEM_PC_FEAT = [
    'sem_pc_0', 'sem_pc_1', 'sem_pc_2', 'sem_pc_4'
]

# first ablation
# EXPERIMENTS = {
#     "core_only": CORE_FEAT,
#     "core_geo": CORE_FEAT + GEO_FEAT,
#     "core_dtw": CORE_FEAT + DTW_FEAT,
#     "core_lags": CORE_FEAT + LAG_FEAT,
#     "core_sim25": CORE_FEAT + SIM25_FEAT,
#     "core_dom": CORE_FEAT + DOM_FEAT,
#     "core_search": CORE_FEAT + SEARCH_FEAT,
#     "core_sem_pc": CORE_FEAT + SEM_PC_FEAT,
#     "all_features": CORE_FEAT + GEO_FEAT + DTW_FEAT + LAG_FEAT + SIM25_FEAT + DOM_FEAT + SEARCH_FEAT + SEM_PC_FEAT
# }

# second ablation
EXPERIMENTS = {
    "core_geo_search": CORE_FEAT + GEO_FEAT + SEARCH_FEAT,
    "core_geo_sem_pc": CORE_FEAT + GEO_FEAT + SEM_PC_FEAT,
}

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def get_scaler_for_experiment(exp_name: str, graph_folder: str, time_series_path: str):
    """Fits a scaler specifically for the feature set of a given experiment."""
    if exp_name not in EXPERIMENTS:
        raise ValueError(f"Experiment {exp_name} not defined in EXPERIMENTS config.")

    features = EXPERIMENTS[exp_name]
    print(f"  → Loading data and fitting scaler for {len(features)} features...")

    # Load Data
    df = pd.read_parquet(time_series_path)

    # Preprocessing (Geo IDs)
    df["detected_city"] = df["detected_city"].astype(str).replace('None', 'Unknown').fillna('Unknown')
    df["detected_country"] = df["detected_country"].astype(str).replace('None', 'Unknown').fillna('Unknown')
    df["detected_continent"] = df["detected_continent"].astype(str).replace('None', 'Unknown').fillna('Unknown')

    city_to_id = {name: i for i, name in enumerate(sorted(df["detected_city"].unique()), start=1)}
    country_to_id = {name: i for i, name in enumerate(sorted(df["detected_country"].unique()), start=1)}
    continent_to_id = {name: i for i, name in enumerate(sorted(df["detected_continent"].unique()), start=1)}

    df["detected_city_id"] = df["detected_city"].map(city_to_id).astype("int32")
    df["detected_country_id"] = df["detected_country"].map(country_to_id).astype("int32")
    df["detected_continent_id"] = df["detected_continent"].map(continent_to_id).astype("int32")

    # Parse Weeks
    if df['week'].dtype == object and df['week'].astype(str).str.contains('-').any():
        parts = df['week'].astype(str).str.split('-', expand=True)
        df['week'] = pd.to_numeric(parts[1]) * 100 + pd.to_numeric(parts[0])
    else:
        df['week'] = pd.to_numeric(df['week'])

    # Identify Training Weeks
    weeks = np.array(sorted(df['week'].unique()))
    trainval_weeks = weeks[:-TEST_WEEKS_LAST]
    split_idx = int(len(trainval_weeks) * (1 - VAL_RATIO))
    train_weeks = trainval_weeks[:split_idx]

    # Filter and Apply Log1p
    df_train = df[df['week'].isin(train_weeks)].copy()
    target_idx = features.index(TARGET_COL)

    X_flat = df_train[features].values
    X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

    scaler = StandardScaler()
    scaler.fit(X_flat)

    return scaler, target_idx

def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
    """Unscales tensors and computes per-node (keyword) metrics."""
    if isinstance(preds, torch.Tensor): preds = preds.cpu().numpy()
    if isinstance(targets, torch.Tensor): targets = targets.cpu().numpy()

    # Reshape to flat for scaler
    preds_flat = preds.reshape(-1, 1)
    targets_flat = targets.reshape(-1, 1)

    num_features = scaler.mean_.shape[0]
    dummy_preds = np.zeros((len(preds_flat), num_features))
    dummy_targets = np.zeros((len(targets_flat), num_features))

    dummy_preds[:, target_col_idx] = preds_flat[:, 0]
    dummy_targets[:, target_col_idx] = targets_flat[:, 0]

    # Inverse transform
    real_preds = np.expm1(scaler.inverse_transform(dummy_preds)[:, target_col_idx])
    real_targets = np.expm1(scaler.inverse_transform(dummy_targets)[:, target_col_idx])
    real_preds = np.maximum(real_preds, 0.0)

    try:
        # Reshape to [Time, Nodes]
        P = real_preds.reshape(-1, NUM_NODES)
        A = real_targets.reshape(-1, NUM_NODES)

        # Per-node RMSE
        node_rmse = np.sqrt(np.mean((P - A) ** 2, axis=0))

        # Per-node SMAPE
        numerator = np.abs(P - A)
        denominator = (np.abs(P) + np.abs(A)) / 2.0
        node_smape = 100 * np.mean(numerator / (denominator + 1e-8), axis=0)

        return {
            'avg_smape': np.mean(node_smape),
            'med_smape': np.median(node_smape),
            'std_smape': np.std(node_smape),
            'avg_rmse': np.mean(node_rmse),
            'med_rmse': np.median(node_rmse),
            'std_rmse': np.std(node_rmse)
        }
    except Exception as e:
        print(f"    ! Reshape error: {e}")
        return None

# =============================================================================
# 3. MAIN EXECUTION
# =============================================================================

def main():
    print("="*80)
    print("ABLATION STUDY: POST-PROCESSING TENSORS")
    print("="*80)

    results_list = []

    # Iterate over experiment folders
    for exp_name in sorted(EXPERIMENTS.keys()):
        exp_dir = os.path.join(TENSOR_ROOT, f"exp_{exp_name}")
        if not os.path.exists(exp_dir):
            print(f"Skipping {exp_name}: Directory not found at {exp_dir}")
            continue

        print(f">>> Processing Experiment: {exp_name}")

        try:
            scaler, target_idx = get_scaler_for_experiment(exp_name, GRAPH_FOLDER_PATH, TIME_SERIES_CSV_PATH)
        except Exception as e:
            print(f"  FAILED to initialize scaler for {exp_name}: {e}")
            continue

        all_files = os.listdir(exp_dir)
        pred_files = [f for f in all_files if f.endswith('_predictions.pt')]

        if not pred_files:
            print("  No prediction tensors found in this directory.")
            continue

        print(f"  Found {len(pred_files)} prediction files.")

        for p_file in sorted(pred_files):
            t_file = p_file.replace('_predictions.pt', '_targets.pt')
            if t_file not in all_files:
                print(f"  ! Missing targets for {p_file}, skipping.")
                continue

            # Extract model and horizon using regex
            # Expected format: {model}_{exp}_h{H}_predictions.pt
            match = re.search(r'(.+)_' + re.escape(exp_name) + r'_h(\d+)_predictions\.pt', p_file)
            if match:
                model_name = match.group(1)
                horizon = int(match.group(2))
            else:
                # Fallback for unexpected naming
                model_name, horizon = p_file.replace("_predictions.pt", ""), "unknown"

            print(f"    Processing {model_name} (H={horizon})...")

            try:
                preds = torch.load(os.path.join(exp_dir, p_file), map_location='cpu')
                targets = torch.load(os.path.join(exp_dir, t_file), map_location='cpu')

                metrics = calculate_advanced_metrics(preds, targets, scaler, target_idx)

                if metrics:
                    results_list.append({
                        'Experiment': exp_name,
                        'Horizon': horizon,
                        'Model': model_name,
                        **metrics
                    })
            except Exception as e:
                print(f"    ! Error processing {p_file}: {e}")

    # Export Final Results
    if results_list:
        output_file = "ablation_detailed_results.csv"
        df_final = pd.DataFrame(results_list)

        # Sort for readability
        df_final = df_final.sort_values(['Experiment', 'Horizon', 'avg_smape'])

        df_final.to_csv(output_file, index=False)
        print("" + "="*80)
        print(f"SUCCESS: Results saved to {output_file}")
        print("="*80)
        print(df_final.head(10).to_string(index=False))
    else:
        print("No valid results were processed.")

if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ABLATION STUDY: POST-PROCESSING TENSORS
>>> Processing Experiment: core_city
  → Loading data and fitting scaler for 270 features...
  Found 12 prediction files.
    Processing AGCRN (H=12)...
    Processing AGCRN (H=1)...
    Processing AGCRN (H=6)...
    Processing DCRNN (H=12)...
    Processing DCRNN (H=1)...
    Processing DCRNN (H=6)...
    Processing GConvLSTM (H=12)...
    Processing GConvLSTM (H=1)...
    Processing GConvLSTM (H=6)...
    Processing GraphWaveNet (H=12)...
    Processing GraphWaveNet (H=1)...
    Processing GraphWaveNet (H=6)...
>>> Processing Experiment: core_continent
  → Loading data and fitting scaler for 9 features...
  Found 12 prediction files.
    Processing AGCRN (H=12)...
    Processing AGCRN (H=1)...
    Processing AGCRN (H=6)...
    Processing DCRNN (H=12)...
    Processing DCRNN (H=1)...
    Processing DCRNN (H=6)...
    P

In [ ]:
# =============================================================================
# Ablation Study: Tensors to Detailed Results (v3 - Dummy Ready)
# =============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import re
from typing import Dict, List, Tuple, Any
from sklearn.preprocessing import StandardScaler

# Mount Google Drive
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# =============================================================================
# 1. CONFIGURATION & FEATURE DISCOVERY
# =============================================================================

TENSOR_ROOT = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/tensors"
TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/sem_dtw_geo_enriched_dummies.parquet"
GRAPH_FOLDER_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn"

TARGET_COL = 'cpc_week'
TEST_WEEKS_LAST = 12
VAL_RATIO = 0.25
NUM_NODES = 1811

# --- Dynamic Feature Discovery ---
_temp_df = pd.read_parquet(TIME_SERIES_CSV_PATH, columns=None)
all_cols = _temp_df.columns.tolist()

# Define Groups based on columns actually present in the file
CORE_FEAT = ['impressions_sum', 'cpc_week']
GEO_CONTINENT_FEAT = [c for c in all_cols if c.startswith('detected_continent_')]
GEO_COUNTRY_FEAT = [c for c in all_cols if c.startswith('detected_country_')]
GEO_CITY_FEAT = [c for c in all_cols if c.startswith('detected_city_')]
SEARCH_FEAT = ['n_st_branded_search', 'n_st_generic_search']

# Fallback for Semantic PC (using neighbors since PC is missing in new parquet)
SEM_PC_FEAT = [c for c in all_cols if c.startswith('semantic_neighbour_')][:4]
if not SEM_PC_FEAT:
    SEM_PC_FEAT = ['sem_pc_0', 'sem_pc_1', 'sem_pc_2', 'sem_pc_4'] # Keep original if found

EXPERIMENTS = {
    "core_continent": CORE_FEAT + GEO_CONTINENT_FEAT,
    "core_country": CORE_FEAT + GEO_COUNTRY_FEAT,
    "core_city": CORE_FEAT + GEO_CITY_FEAT,
    "core_continent_search": CORE_FEAT + GEO_CONTINENT_FEAT + SEARCH_FEAT,
    "core_country_search": CORE_FEAT + GEO_COUNTRY_FEAT + SEARCH_FEAT,
    "core_continent_sempc": CORE_FEAT + GEO_CONTINENT_FEAT + SEM_PC_FEAT,
}

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def get_scaler_for_experiment(exp_name: str, time_series_path: str):
    features = EXPERIMENTS[exp_name]

    # Check for missing columns before loading
    missing = [c for c in features if c not in all_cols]
    if missing:
        raise ValueError(f"Experiment '{exp_name}' has missing columns in Parquet: {missing}")

    df = pd.read_parquet(time_series_path, columns=features + ['week'])

    # Parse Weeks
    if df['week'].dtype == object:
        parts = df['week'].astype(str).str.split('-', expand=True)
        df['week'] = pd.to_numeric(parts[1]) * 100 + pd.to_numeric(parts[0])
    else:
        df['week'] = pd.to_numeric(df['week'])

    # Split (Matching Training Logic)
    weeks = np.array(sorted(df['week'].unique()))
    trainval_weeks = weeks[:-TEST_WEEKS_LAST]
    split_idx = int(len(trainval_weeks) * (1 - VAL_RATIO))
    train_weeks = trainval_weeks[:split_idx]

    df_train = df[df['week'].isin(train_weeks)].copy()
    target_idx = features.index(TARGET_COL)

    # Force float32 to avoid 'no callable log1p' error
    X_flat = df_train[features].values.astype(np.float32)
    X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

    scaler = StandardScaler()
    scaler.fit(X_flat)
    return scaler, target_idx

def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
    if isinstance(preds, torch.Tensor): preds = preds.cpu().numpy()
    if isinstance(targets, torch.Tensor): targets = targets.cpu().numpy()

    # Flatten for inverse transform
    p_flat = preds.reshape(-1, 1)
    t_flat = targets.reshape(-1, 1)

    num_features = scaler.mean_.shape[0]
    d_p = np.zeros((len(p_flat), num_features))
    d_t = np.zeros((len(t_flat), num_features))

    d_p[:, target_col_idx] = p_flat[:, 0]
    d_t[:, target_col_idx] = t_flat[:, 0]

    # Inverse transform and Reverse Log1p
    real_preds = np.expm1(scaler.inverse_transform(d_p)[:, target_col_idx])
    real_targets = np.expm1(scaler.inverse_transform(d_t)[:, target_col_idx])
    real_preds = np.maximum(real_preds, 0.0)

    try:
        P = real_preds.reshape(-1, NUM_NODES)
        A = real_targets.reshape(-1, NUM_NODES)

        node_rmse = np.sqrt(np.mean((P - A) ** 2, axis=0))
        num = np.abs(P - A)
        den = (np.abs(P) + np.abs(A)) / 2.0
        node_smape = 100 * np.mean(num / (den + 1e-8), axis=0)

        return {
            'avg_smape': np.mean(node_smape), 'med_smape': np.median(node_smape), 'std_smape': np.std(node_smape),
            'avg_rmse': np.mean(node_rmse), 'med_rmse': np.median(node_rmse), 'std_rmse': np.std(node_rmse)
        }
    except Exception as e:
        return None

# =============================================================================
# 3. MAIN EXECUTION
# =============================================================================

results_list = []
for exp_name in sorted(EXPERIMENTS.keys()):
    exp_dir = os.path.join(TENSOR_ROOT, f"exp_{exp_name}")
    if not os.path.exists(exp_dir): continue

    print(f"\n>>> Processing: {exp_name}")
    try:
        scaler, target_idx = get_scaler_for_experiment(exp_name, TIME_SERIES_CSV_PATH)
    except Exception as e:
        print(f"  FAILED: {e}")
        continue

    all_files = os.listdir(exp_dir)
    pred_files = [f for f in all_files if f.endswith('_predictions.pt')]

    for p_file in sorted(pred_files):
        t_file = p_file.replace('_predictions.pt', '_targets.pt')
        if t_file not in all_files: continue

        # Extract model and horizon: {Model}_{Exp}_h{H}_predictions.pt
        match = re.search(r'(.+)_' + re.escape(exp_name) + r'_h(\d+)_predictions\.pt', p_file)
        model_name, horizon = (match.group(1), int(match.group(2))) if match else (p_file, "??")

        print(f"    - {model_name} (H={horizon})")
        preds = torch.load(os.path.join(exp_dir, p_file), map_location='cpu')
        targets = torch.load(os.path.join(exp_dir, t_file), map_location='cpu')

        metrics = calculate_advanced_metrics(preds, targets, scaler, target_idx)
        if metrics:
            results_list.append({'Experiment': exp_name, 'Horizon': horizon, 'Model': model_name, **metrics})

if results_list:
    df_final = pd.DataFrame(results_list).sort_values(['Experiment', 'Horizon', 'avg_smape'])
    df_final.to_csv("ablation_detailed_results_v3.csv", index=False)
    print("\n" + "="*50 + "\nSUCCESS: Saved to ablation_detailed_results_v3.csv\n" + "="*50)
    print(df_final.head(10))


>>> Processing: core_city
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (H=6)
    - GConvLSTM (H=12)
    - GConvLSTM (H=1)
    - GConvLSTM (H=6)
    - GraphWaveNet (H=12)
    - GraphWaveNet (H=1)
    - GraphWaveNet (H=6)

>>> Processing: core_continent
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (H=6)
    - GConvLSTM (H=12)
    - GConvLSTM (H=1)
    - GConvLSTM (H=6)
    - GraphWaveNet (H=12)
    - GraphWaveNet (H=1)
    - GraphWaveNet (H=6)

>>> Processing: core_continent_search
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (H=6)
    - GConvLSTM (H=12)
    - GConvLSTM (H=1)
    - GConvLSTM (H=6)
    - GraphWaveNet (H=12)
    - GraphWaveNet (H=1)
    - GraphWaveNet (H=6)

>>> Processing: core_continent_sempc
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (